In [13]:
# Fast Parallel + Vectorised Script with Progress Bar + Logging + Timers

import pandas as pd
import os
import numbers
import time
import logging
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
os.chdir("/tmp")
# --- STEP 1: Load both CSV rule files with actual column names ---

null_path = "helper_files/null caddphredd.csv"
nonnull_path = "helper_files/cadd_phredd.csv"

# Load and clean null CADD_PHRED rules
null_df = pd.read_csv(null_path)
null_df = null_df.rename(columns={
    'Unique Null Consequences': 'Consequence',
    'CADD_PHRED imputed': 'CADD_PHRED'
})
null_df = null_df.applymap(lambda x: x.strip().lower() if isinstance(x, str) else x)
null_df['CADD_PHRED'] = null_df['CADD_PHRED'].replace({'low': 'MIN', 'max': 'MAX', '0': 0})
null_df['CADD_PHRED'] = pd.to_numeric(null_df['CADD_PHRED'], errors='ignore')

# Load and clean non-null CADD_PHRED rules
nonnull_df = pd.read_csv(nonnull_path)
nonnull_df = nonnull_df.rename(columns={
    'CADD_PHEDD': 'CADD_PHRED'
})
nonnull_df = nonnull_df.applymap(lambda x: x.strip().lower() if isinstance(x, str) else x)
nonnull_df['CADD_PHRED'] = nonnull_df['CADD_PHRED'].replace({'low': 'MIN', 'max': 'MAX', '0': 0})
nonnull_df['CADD_PHRED'] = pd.to_numeric(nonnull_df['CADD_PHRED'], errors='ignore')

# --- STEP 2: Convert to rule dictionaries ---

def build_rule_dict(df):
    rule_dict = {}
    numeric_vals = pd.to_numeric(df['CADD_PHRED'], errors='coerce')
    min_val = numeric_vals.min()
    max_val = numeric_vals.max()
    for _, row in df.iterrows():
        cons = row['Consequence']
        val = row['CADD_PHRED']
        if isinstance(val, str):
            rule_dict[cons] = val.upper()  # 'MIN' or 'MAX'
        else:
            rule_dict[cons] = val
    return rule_dict

null_rules = build_rule_dict(null_df)
nonnull_rules = build_rule_dict(nonnull_df)

# --- STEP 3: Fast Parallel + Vectorised Processing ---

input_folder = "Individual_data_Uniti_generated_features_withDC_with_Adjusted AF for CRI_cohort_new"
output_folder = os.path.join(input_folder, "cadd_phredd_corrections_final_Uniti_new")
os.makedirs(output_folder, exist_ok=True)

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

def process_file(file_path):
    start = time.time()
    file = os.path.basename(file_path)
    try:
        df = pd.read_csv(file_path, sep="\t", low_memory=False)
    except Exception as e:
        return f"❌ Error reading {file}: {e}"

    if 'CADD_PHRED' not in df.columns or 'Consequence' not in df.columns:
        return f"⚠️ Skipping {file}: Missing required columns."

    df['CADD_PHRED'] = pd.to_numeric(df['CADD_PHRED'], errors='coerce')
    file_min = df['CADD_PHRED'].min(skipna=True)
    file_max = df['CADD_PHRED'].max(skipna=True)

    if pd.isna(file_min) or pd.isna(file_max):
        return f"⚠️ Skipping {file}: No numeric values for MIN/MAX."

    updated = False

    # Apply null rules
    null_mask = df['CADD_PHRED'].isna()
    if null_mask.any():
        for cons, rule in null_rules.items():
            cons_mask = df['Consequence'].str.strip().str.lower() == cons
            mask = null_mask & cons_mask
            if mask.any():
                if rule == 'MIN':
                    df.loc[mask, 'CADD_PHRED'] = file_min
                elif rule == 'MAX':
                    df.loc[mask, 'CADD_PHRED'] = file_max
                elif rule == 0:
                    df.loc[mask, 'CADD_PHRED'] = 0
                elif isinstance(rule, numbers.Number):
                    df.loc[mask, 'CADD_PHRED'] = rule
                updated = True

    # Apply non-null rules
    nonnull_mask = df['CADD_PHRED'].notna()
    if nonnull_mask.any():
        for cons, rule in nonnull_rules.items():
            cons_mask = df['Consequence'].str.strip().str.lower() == cons
            mask = nonnull_mask & cons_mask
            if mask.any():
                if rule == 'MIN':
                    df.loc[mask, 'CADD_PHRED'] = file_min
                elif rule == 'MAX':
                    df.loc[mask, 'CADD_PHRED'] = file_max
                elif rule == 0:
                    df.loc[mask, 'CADD_PHRED'] = 0
                elif isinstance(rule, numbers.Number):
                    df.loc[mask, 'CADD_PHRED'] = rule
                updated = True

    elapsed = time.time() - start

    if updated:
        corrected_path = os.path.join(output_folder, file)
        df.to_csv(corrected_path, sep="\t", index=False)
        return f"🔧 Corrected and saved: {file} in {elapsed:.2f} sec"
    else:
        return f"✅ No changes needed: {file} ({elapsed:.2f} sec)"

# --- Run all files in parallel ---
if __name__ == "__main__":
    files = [os.path.join(input_folder, f) for f in os.listdir(input_folder) if f.endswith(".tsv")]

    with ProcessPoolExecutor() as executor:
        futures = [executor.submit(process_file, f) for f in files]

        # tqdm over futures directly (updates as tasks complete)
        for result in tqdm(
            map(lambda f: f.result(), futures),
            total=len(futures),
            desc="Processing files",
            dynamic_ncols=True,
            unit="file",
            smoothing=0.3
        ):
            logger.info(result)

    logger.info("🎉 All files processed successfully!")

/tmp/ipykernel_302864/154820203.py:22: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  null_df = null_df.applymap(lambda x: x.strip().lower() if isinstance(x, str) else x)
/tmp/ipykernel_302864/154820203.py:24: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  null_df['CADD_PHRED'] = pd.to_numeric(null_df['CADD_PHRED'], errors='ignore')
/tmp/ipykernel_302864/154820203.py:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  nonnull_df = nonnull_df.applymap(lambda x: x.strip().lower() if isinstance(x, str) else x)
/tmp/ipykernel_302864/154820203.py:33: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  nonnull_df['CADD_PHRED'] = pd.to_numeric(nonnull_df['CADD_PHRED'], errors='ignore')
Processing f

In [15]:
import os
import pandas as pd

# Folder created by your script
corrected_folder = "Individual_data_Uniti_generated_features_withDC_with_Adjusted AF for CRI_cohort_new"

def load_all_patients(folder):
    all_data = []
    file_count = 0

    for file in os.listdir(folder):
        if file.endswith(".tsv"):
            path = os.path.join(folder, file)
            try:
                df = pd.read_csv(path, sep="\t", low_memory=False)

                # add patient id column
                df["Patient_ID"] = file

                all_data.append(df)
                file_count += 1

            except Exception as e:
                print(f"❌ Failed to read {file}: {e}")

    if not all_data:
        print("No files loaded.")
        return None

    combined = pd.concat(all_data, ignore_index=True)
    print(f"\nLoaded {file_count} patients")
    print(f"Total variants: {len(combined)}")

    return combined


# -------- RUN ----------
combined_df = load_all_patients(corrected_folder)

# Preview
if combined_df is not None:
    print("\nColumns:")
    print(combined_df.columns.tolist())

    print("\nFirst 5 rows:")
    print(combined_df.head())

    print("\nBasic stats:")
    print(combined_df.describe(include='all'))



Loaded 166 patients
Total variants: 836

Columns:
['CHROM', 'POS', 'REF', 'ALT', 'Gene', 'SYMBOL', 'Consequence', 'CADD_PHRED', 'AF', 'GT', 'gene_id', 'start', 'end', 'gene_length', 'gene_score', 'Phenotype', 'PH', 'DCs_score', 'Adjusted AF for CRI', 'Patient_ID']

First 5 rows:
   CHROM       POS REF ALT             Gene    SYMBOL  \
0   chr1  40307477   T   C  ENSG00000049089    COL9A2   
1   chr4  42963132   G   A  ENSG00000215203    GRXCR1   
2   chr5  69419483   C   T  ENSG00000152939  MARVELD2   
3  chr16   4614411   T  TG  ENSG00000102858     MGRN1   
4  chr22  50548858   A   G  ENSG00000130487   KLHDC7B   

                              Consequence  CADD_PHRED        AF   GT  \
0                        missense_variant      14.310  0.001166  0/1   
1  missense_variant&splice_region_variant      34.000  0.001166  0/1   
2                        missense_variant       1.147  0.001166  0/1   
3                   upstream_gene_variant       6.823  0.001166  0/1   
4               